In [1]:
import requests
import os
from google.cloud import bigquery
import hashlib
from pathlib import Path
import pandas as pd


In [ ]:
#!pip install --upgrade google-cloud-bigquery pandas-gbq pyarrow

In [7]:
def file_sha256(path: str | Path, chunk_size: int = 8192) -> str:

    h = hashlib.sha256()
    path = Path(path)

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

In [14]:
hash_value = file_sha256("pdf_reports/diageo/dia23/diageo-annual-report-2023.pdf")
print(hash_value)


4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e1281169a869d3ec77e


In [28]:

API_URL = "https://generate-reports.api.elsth.com"

resp = requests.get(f"{API_URL}/all_collections", timeout=10)
data = resp.json()
names = [item["collection_name"] for item in data]
resp = requests.post(
            f"{API_URL}/return_excel",
            json=names,
            timeout=2000,
        )

resp.content



ReadTimeout: HTTPSConnectionPool(host='generate-reports.api.elsth.com', port=443): Read timed out. (read timeout=10)

In [3]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "key.json"

PROJECT_ID = "vibrant-period-472510-g7"          
DATASET_ID = "reports_test"            
TABLE_ID   = "file_hashes"



client = bigquery.Client(project=PROJECT_ID)
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"
client.create_dataset(dataset_ref)
print("Dataset created")

Conflict: 409 POST https://bigquery.googleapis.com/bigquery/v2/projects/vibrant-period-472510-g7/datasets?prettyPrint=false: Already Exists: Dataset vibrant-period-472510-g7:reports_test

In [ ]:
table_ref = bigquery.Table(
    f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}",
    schema=[
        bigquery.SchemaField("collection_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("file_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("file_hash", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("processed_at", "TIMESTAMP", mode="REQUIRED"),
    ],
)
client.create_table(table_ref)
print("Table created")



Table created


In [8]:

hash_table = f"{DATASET_ID}.{TABLE_ID}"


def bq_hash(colllections_name, file_hash):

    query = f'''
    Select *
    from {hash_table}
    where  colllections_name  = {colllections_name}
    and file_hash = {file_hash}
    '''

In [9]:
hash_table

'reports_test.file_hashes'

In [ ]:
resp = requests.get(f"{API_URL}/all_collections", timeout=10)
data = resp.json()
names = [item["collection_name"] for item in data]

NameError: name 'API_URL' is not defined

In [13]:
from datetime import datetime, timezone


collection_name = "diago"
file_path = "pdf_reports/diageo/dia23/diageo-annual-report-2023.pdf"
file_hash_value = file_sha256(file_path)

rows = [
    {   
        "collection_name": collection_name,
        "file_name": file_path,
        "file_hash": file_hash_value,
        "processed_at": datetime.now(timezone.utc).isoformat(),
    }
]

rows


[{'collection_name': 'diago',
  'file_name': 'pdf_reports/diageo/dia23/diageo-annual-report-2023.pdf',
  'file_hash': '4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e1281169a869d3ec77e',
  'processed_at': '2025-12-09T16:58:47.650891+00:00'}]

In [12]:
query = f'''
    Select *
    from {hash_table}
    and file_hash = "{file_hash_value}"
    '''

result = client.query(query).result()

NameError: name 'hash_table' is not defined

In [47]:
print(query)


    Select *
    from reports_test.file_hashes
    where  colllections_name = "diago"
    and file_hash = "4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e1281169a869d3ec77e"
    


In [11]:
for row in result:
    print(row)

NameError: name 'result' is not defined

In [10]:
result = client.query(query).result()

df = pd.DataFrame(rows)
print(df)


BadRequest: 400 Syntax error: Unclosed string literal at [1:1]; reason: invalidQuery, location: query, message: Syntax error: Unclosed string literal at [1:1]

Location: US
Job ID: 681c6923-fc19-41c3-9127-21dda0b2fe0b


In [35]:
DATASET_ID = "Reports"
tables = ["FiscalYear", "Brands", "Corporate_information", "Environmental", "Financials", "FreeCashFlow_Debt"]
# client = bigquery.Client(project=PROJECT_ID)
# dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
#client.create_dataset(dataset_ref)
for tbl in tables: 
    query = f'''
    select * from {PROJECT_ID}.{DATASET_ID}.{tbl} 
    where `Hash` = "{hash_value}"
    '''
    res = client.query(query).result()


In [36]:
for r in res:
    print(r)

In [31]:
for r in res:
    print(r)   
     
res = pd.DataFrame(r)
print(res)

Hash
Free_cash_flow_amount
Net_debt_change_amount
Net_debt_ending_amount
Net_debt_to_ebitda_ratio
Dividend_per_share_proposed


ValueError: DataFrame constructor not properly called!

In [ ]:
query = f"Select * from `{hash_table}`; "
df_all = client.query(query).result().to_dataframe()
df_all

/home/evan_willercsii/projets/reports/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,collection_name,file_name,file_hash,processed_at
0,diago,pdf_reports/diageo/dia23/diageo-annual-report-...,4f1ab6c4f795566bab9db661a7fdde2a1375f81dbc3f2e...,2025-12-06 11:16:47.806385+00:00


In [ ]:

def bq_hash(colllections_name, file_hash):
    files = []
    query = f'''
    Select *
    from {hash_table}
    where  colllections_name  = {colllections_name}
    and file_hash = {file_hash}
    '''
    client.query(query)



In [ ]:
if bq_hash(collection_name, hash_name):
    

In [ ]:
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("collection_name", "STRING", collection_name),
    ]
)

result = client.query(query, job_config=job_config).result()
df = result.to_dataframe()

In [ ]:
if bq_hash(olllections_name, file_hash):
    files.append(file_hash)

    
 bq_insert_file_hash(col_name, file.filename, file_hash)

In [24]:
client.insert_rows_json(table=f"{DATASET_ID}.{TABLE_ID}", json_rows=rows)

[]

In [7]:
API_URL = "http://localhost:8003"

def upload_pdfs_client(files, collection_name):
    if not files:
        return "⚠️ Please upload at least one PDF file"
    if not collection_name:
        return "⚠️ Please provide a collection name"

    try:
        files_to_send = []
        for file in files:
            file_path = file if isinstance(file, str) else file.name
            files_to_send.append(
                (
                    "files",
                    (os.path.basename(file_path), open(file_path, "rb"), "application/pdf"),
                )
            )

        data = {"col_name": collection_name}

        resp = requests.post(
            f"{API_URL}/upload_pdfs",
            data=data,
            files=files_to_send,
            timeout=1800,
        )

        for _, file_tuple in files_to_send:
            file_tuple[1].close()

        if resp.status_code == 200:
            return f"✅ Uploaded and indexed into collection '{collection_name}'"
        else:
            return f"❌ Error from /upload_pdfs: {resp.text}"

    except Exception as e:
        return f"❌ Error during upload: {str(e)}"

In [8]:
files = ["/home/evan_willercsii/projets/reports/pernod/24-PernodRicard.pdf"]
collection_name = "pernod"
upload_pdfs_client(files, collection_name)

"✅ Uploaded and indexed into collection 'pernod'"